In [0]:
df_spark = spark.read.table("workspace.churn_schema.customer_churn_dataset_training_master")
df_spark.show(5)

In [0]:
df = df_spark.toPandas()
df.head()

In [0]:
df.info()

In [0]:
#checking for null
def check_for_null(df):
    missing_counts=df.isnull().sum()
    missing_cols = {col: cnt for col, cnt in missing_counts.items() if cnt > 0}
    if missing_cols:
        print("Missing values detected:")
        for col, cnt in missing_cols.items():
            print(f" - Column '{col}' has {cnt} missing value(s)")
    else:
        print("No missing values detected!")
    
    return missing_cols

def check_for_numeric_values(numeric_cols):
    if numeric_cols is None:
        numeric_cols= df.select_dtypes(include=['float64', 'int64']).columns
    negative_col={}
    for col in numeric_cols:
        neg_count= (df[col]<0).sum()
        print(neg_count)
        if neg_count>0:
               negative_col[col] = neg_count

    if negative_col:
          print("Negative values detected:")
    for col, count in negative_col.items():
            print(f" - Column '{col}' has {count} negative value(s)")
    else:
          print("No negative values detected in numeric columns!")

    return negative_col

def check_categorical_data(df,allowed_values):
    
    categorical_data=df.select_dtypes(include=['object']).columns
    issues={}
    for cols in categorical_cols:
        unique_col= set(df[cols].dropna().unique())   
        if allowed_values and cols in allowed_values:
            invalid_vals = unique_col - set(allowed_values[cols])
            if invalid_vals:
                issues[cols] = invalid_vals
        else:
            print(f"Column '{cols}' unique values ({len(unique_col)}): {unique_col}")  
        if issues:
            print(f"issues'{issues}")    


In [0]:
numeric_cols = ['CustomerID','Age','Tenure','Usage Frequency','Support Calls',
                'Payment Delay','Total Spend','Last Interaction','Churn']

allowed_values = {
    "Gender": ["Male", "Female"],
    "Subscription Type": ["Basic","Standard","Premium", "Pro"],   
    "Contract Length": ["Annual", "Monthly", "Quarterly", "Yearly"] 
}
print(check_categorical_data(df,allowed_values=allowed_values))


In [0]:
def df_transform(df):
    nurmeric_columns=df.select_dtypes(include=['float64']).columns
    for col in nurmeric_columns:
        df[col]=df[col].fillna(df[col].median())
    print("finished")
    categorical_cols = df.select_dtypes(include=['object']).columns
    for col in categorical_cols:
       if not df[col].mode().empty:
        df[col] = df[col].fillna(df[col].mode()[0])
       else:
        df[col] = df[col].fillna("Unknown")

    print("Finished categorical fill")

# Fix types
    df['CustomerID'] = df['CustomerID'].round().astype('Int64')   # nullable int
    df['Churn'] = df['Churn'].fillna(0).astype(int)    

    return df
         

In [0]:
df=df_transform(df)
df = df.rename(columns=lambda x: x.strip().replace(" ", "_").replace("-", "_"))



In [0]:
df.info()
df_spark = spark.createDataFrame(df)

In [0]:

df_spark.write.format("delta").mode("overwrite").saveAsTable("workspace.churn_schema.churn_dataset")
